# 19 — Inventário e recuperação S3 limitada

Implementação preparada em 2026-10-15. Arquivos gerados e S3 simulado, sem dados
nativos ou execução Docker nesta preparação. O autor depois confirmou inventário completo e recuperação de dois documentos
no RustFS; resultados relatados, sem inspeção direta dos bytes pelo assistente. O [runbook](../../docs/operations/STORAGE.md) é a referência dos comandos.


## Problema e decisão

Datasets e resultados locais precisam de inventário, endereço e recuperação
verificáveis. Subir um servidor de objetos não prova que os dados foram preservados.
Começamos por metadados de todas as famílias de `data/`; depois, um conjunto
explícito de no máximo 64 arquivos/64 MiB. Esse orçamento valida o contrato sem
repetir o replay ou deslocar gigabytes antes de medir espaço e custo.

RustFS é o backend S3 local escolhido. O cliente usa imagem oficial AWS CLI
fixada por digest, fora das dependências congeladas do experimento. DuckDB
permanece motor analítico; PostgreSQL é o destino escolhido do futuro servidor
MLflow. A troca do backend preservará IDs/runs/URIs em etapa própria. A feature
store depende de reutilização e requisitos de disponibilidade demonstrados.


## Contrato e execução gradual

1. Executar somente `poetry run python -m scripts.storage.archive inventory`.
   Verificar famílias, bytes, espaço livre, itens não classificados e `complete`.
   Não lê payloads e não tem teto de tempo/arquivos; erro/symlink deixam lacuna.
2. Escolher arquivos pequenos de uma release/avaliação concluída no inventário.
   `prepare --include <path>` cria cópia local e manifesto; banco ativo é recusado.
3. Iniciar apenas RustFS no perfil opcional, conferir saúde e executar `publish`.
   Repetir a publicação deve reutilizar todos os objetos com zero uploads.
4. `restore` baixa em novo diretório e confere paths, identidade, tamanho e SHA-256
   antes de publicar a restauração. Não sobrescreve o original nem uma restauração.

[Comandos completos](../../docs/operations/STORAGE.md#piloto-implementado--execução-gradual).
Os serviços de observabilidade não precisam ser reiniciados. Não há DELETE
automático, bucket AWS, upload do `data/` inteiro ou commit nesta rotina.


## Evidência e limites

Testes pytest com filesystem real temporário e S3 simulado validam integridade,
retomada, publicação do manifesto por último, corrupção, limites e caminhos.
O autor validou a execução real no RustFS: primeira publicação com três uploads,
segunda com zero uploads/três objetos reutilizados e restauração de dois arquivos
(17.559 bytes). O inventário contou 1.643 arquivos/2,92 GiB, sem ocorrências.
Os dez arquivos foram depois identificados, e a release completa passou
publicação/reenvio/restauração: quatro arquivos, 735.485 bytes. Foram recebidas saídas de terminal,
sem bytes completos dos relatórios/objetos para recalcular hashes independentemente.
[Evidência relatada](../../references/evidence/storage_pilot_author_validation_2026-10-15.json).

Round-trip de bytes não confirma recarga do modelo, portabilidade das URIs,
backup de SQLite, cobertura Bronze/Silver/Gold ou recuperação após perda do disco.
As próximas ampliações dependem do inventário e do resultado do piloto.
KinD + Helm continua como destino após persistência/restauração e rollback.

[Roadmap](../../docs/project/ROADMAP.md), [stakeholders](../../docs/project/STAKEHOLDERS.md)
e [checks de preparação](../../references/evidence/storage_pilot_preparation_2026-10-15.json).


## Release restaurada: separar bytes de execução

O autor confirmou cinco objetos enviados/verificados e depois reutilizados sem
novos uploads. A restauração trouxe três arquivos da release e seu recibo
de exportação: quatro arquivos, 735.485 bytes. [Evidência relatada](../../references/evidence/storage_release_archive_author_validation_2026-10-15.json).

`tests/storage/test_restored_release.py` reutiliza o loader existente e o pin
histórico externo ao bundle. Os controles gerados passam sem originais e falham
com corrupção/pin/score/identidade divergentes. O autor aprovou o caso nativo:
1 passed/6 deselected em 0,32 s; score `0.4178630794049542`, erro zero, readiness
200 e entrada inválida 422. Identidade histórica preservada e leitura somente
da release restaurada. [Evidência](../../references/evidence/storage_release_reload_author_validation_2026-10-15.json).
O teste verifica readiness, score histórico e entrada inválida via TestClient,
sem porta TCP ou serviço novo. [Comando](../../docs/operations/STORAGE.md#release-completa-recuperação-de-bytes-e-recarga).

No provisionamento Helm, os serviços persistentes terão PVCs com localização
física e retenção declaradas, além de recuperação testada após recriar o cluster.
Isso permanece planejado; Compose é a bancada atual.


## Bronze completa: publicação e recuperação verificadas

O conjunto observado ultrapassa o piloto de 64 arquivos/64 MiB. Criamos o perfil
`fraud_bronze_bundle_v1` (512 arquivos/256 MiB) para um snapshot inteiro, com
manifesto/auditorias originais e cópia do inventário versionado. O formato e IDs
dos bundles válidos do piloto permanecem preservados.

`python -m scripts.storage.bronze_archive prepare` reutiliza o verificador
Bronze existente, mantém arquivos como bytes opacos e confere o staging antes
de publicar localmente. O bundle se verifica sem fonte/inventário originais.
Os controles usam 70 partições geradas e bloqueiam rede/desserialização; cobrem
corrupção, cobertura, locks, bancos, auditorias mutantes e arquivos extras.
Tox relacionado: 177 passed, 1 skipped; não é execução nativa da Bronze.

O autor aprovou o preparo nativo: **183 partições, 188 arquivos,
107.309.243 bytes**, bundle
`bca0bdcbd60853db575dcdd5af986fd7608057c69fb368fd5fa0007aa62bc4c0`.
S3 não foi acessado pelo preparo.
[Evidência](../../references/evidence/bronze_archive_author_validation_2026-10-15.json).

A transferência implementada usa `scripts.storage.bronze_transfer`: um job
Compose/cliente S3 por comando em `fraud-datasets/bronze/v1/<bundle_id>/`.
O SDK tem dependências fixadas fora do lock científico. PUT condicional,
readback SHA-256, manifesto por último e reenvio idempotente; download em
quarentena, seguido do verificador Bronze com o inventário baixado antes de
receber o caminho final. Não consulta os originais nessa restauração.

O autor concluiu o build e aprovou a execução nativa no RustFS:

| Operação | Resultado | Tempo do worker |
| --- | --- | --- |
| Publicação | 189 enviados/conferidos | 3,558664244999818 s |
| Reenvio | Zero uploads; 189 reutilizados/conferidos | 1,025176987001032 s |
| Download | 188 arquivos/107.309.243 bytes; 189 objetos conferidos | 0,8571889770028065 s |

A conferência final no host retornou `restored_verified`, `complete=true`,
**183 partições completas** e inventário incluído; sem leitura da fonte original
ou pickle. O manifesto externo é adicional aos 188 arquivos de conteúdo.
Os tempos acima excluem parte da chamada Poetry/Compose e a verificação Bronze
posterior no host; não há benchmark/SLA nem custo CPU/RAM consolidado.
[Evidência nativa relatada](../../references/evidence/bronze_transfer_author_validation_2026-10-15.json).
Os JSONs foram relatados pelo autor; o assistente não inspecionou o bucket.

O gate de arquivamento/recuperação Bronze está fechado. Os consumidores seguem
nos paths locais; Silver é a próxima ampliação proposta, com contrato/schema,
contagens e proveniência conferidos antes de mudar leitores. Gold e backups dos
bancos seguem com contratos próprios. As fixtures/SDK da preparação permanecem
196 passed/2 skipped no `py314` e quatro aprovados no `bronze-sdk`; não foram
reexecutados para esse fechamento documental. Aplique o fechamento sem repetir
o restore já concluído (seu destino é protegido contra sobrescrita). [Procedimento](../../docs/operations/STORAGE.md#bronze-completa--preparo-offline-antes-da-transferência).


## Silver completa: contrato e contagens acompanham o archive

`scripts.storage.silver_archive` implementa `prepare`, `verify`, `publish` e
`restore`. Inclui `silver_v1/`, suas auditorias em `runs/silver_<run_id>.json`,
inventário e contrato originais; deixa a EDA fora do conjunto. A auditoria do
run criador é obrigatória, e seus bytes históricos são preservados.

Perfil `fraud_silver_bundle_v1`, até 512 arquivos/256 MiB, manifesto até 256 KiB.
O staging e o restore passam por `verify_silver` com as referências arquivadas:
integridade, schema Arrow, cobertura, linhas por partição e contagens SQL.
Não refaz aceitação/normalização Bronze, lê pickle ou reconstrói o dataset.
Proveniência/inputs e auditorias são preservados, sem recomputar a Bronze.

Transporte compartilhado com a Bronze, um job/cliente por comando, prefixo
`fraud-datasets/silver/v1/<bundle_id>/`, manifesto por último e reenvio idempotente.
Download em quarentena; verificação Silver antes do caminho final. Os testes
removem fonte, referências e bundle locais antes do restore. O autor reconstruiu e validou o job para essa versão; mudanças futuras de código
exigem rebuild pelo handshake. Schemas, prefixos e IDs dos bundles históricos
são preservados.

**Preparo e round-trip nativos aprovados pelo autor:** bundle
`f9ec010e67cdd4bd7b8fe992b7503ed4c8a3c0147a63caadf882b8f7db2cf224`.
187 arquivos/40.288.931 bytes (183 Parquets, manifesto Silver, auditoria,
inventário e contrato); manifesto externo adicional, total 188 objetos.

| Operação | Resultado | Tempo do worker |
| --- | --- | --- |
| Publicação | 188 enviados/conferidos | 3,329493249002553 s |
| Reenvio | Zero uploads; 188 reutilizados/conferidos | 0,986083370000415 s |
| Download | 187 arquivos; 188 objetos conferidos | 0,8359925039985683 s |

O host retornou `restored_verified`, `complete=true`, schema/SQL reconciliados:
183 partições, 1.754.155 linhas/IDs distintos, 14.681 fraudes, 1.739.474 genuínas
e 42 valores zero. Contrato/inventário baixados; sem leitura de originais,
Bronze ou pickle. Os tempos do worker excluem a verificação final no host e
parte da chamada Poetry/Compose; não são benchmark ou SLA.
[Evidência nativa relatada](../../references/evidence/silver_archive_author_validation_2026-10-15.json).
O assistente recebeu os JSONs, sem inspecionar bucket, recibos ou arquivos nativos.
Gate de arquivamento/recuperação Silver fechado; próximos conjuntos propostos:
mapear versões/variantes Gold e dependências antes de implementar seu archive.
Os testes da preparação permanecem 235 passed/2 skipped e quatro no SDK;
este fechamento documental não os reexecuta nem repete o restore já existente.

[Runbook](../../docs/operations/STORAGE.md#silver-completa-preparo-e-recuperação-com-contrato) e
[evidência da preparação](../../references/evidence/silver_archive_preparation_2026-10-15.json).
O arquivo local segue sendo consumido; Gold, tracking e Helm/PVCs têm gates próprios.


## Gold original: contrato temporal acompanha a recuperação

O inventário relatado tem 506 arquivos/326.821.198 bytes. `gold_v1` representa
43 arquivos/25.151.651 bytes; desenvolvimento reúne dataset/estudo, modelos e
avaliações são conjuntos próprios. Os paths/contagens são metadados, não prova
de integridade. [Inventário](../../references/evidence/gold_storage_inventory_author_2026-10-15.json).

`python -m scripts.storage.gold_archive prepare` seleciona `gold_v1/`, auditorias
`runs/gold_<run_id>.json` (criador obrigatório), inventário, contratos Silver/Gold
e protocolo temporal. Schema `fraud_gold_bundle_v1`, 512 arquivos/256 MiB,
manifesto externo até 256 KiB. Os bytes de manifests/referências/auditorias
históricas são preservados; variantes de desenvolvimento/estudos ficam em seus gates.

O verificador existente é reaplicado com referências arquivadas: tamanho/hash,
schema Arrow, invariantes das 19 features, datas, IDs únicos e split counts.
Não recalcula features, lê Silver/Bronze/pickle ou executa modelo/novo holdout.
As fixtures removem Gold original, referências e bundle antes do restore.

**Ciclo nativo aprovado pelo autor em 2026-10-15.** Preparo: 48 arquivos,
25.196.448 bytes, uma auditoria e quatro referências. Bundle
`af32af6befe8ad2dba7f36965011ead09d9cdf7c0ac9be81bffcdea92e2e45aa`.
Publicação: 49 objetos enviados e conferidos; reenvio: zero uploads/49 reutilizados.
Restore: `restored_verified`, 48 arquivos, 42 partições, 19 features e 402.877
linhas; treino 268.668, validação 67.255 e teste 66.954. A cópia passou o verificador
com inventário, contratos e protocolo baixados; não leu originais, Bronze/Silver,
recomputou features ou executou modelo. O manifesto externo é o 49º objeto.

Recibos em `data/storage/receipts/`: `gold-prepare-6r5krrua.json`,
`gold-publish-cm0bl7pk.json`, `gold-publish-ul3sv3wv.json` e
`gold-restore-s3yo571_.json`. O assistente recebeu as saídas impressas, sem acessar
bucket/recibos/dados nativos. [Evidência nativa](../../references/evidence/gold_archive_author_validation_2026-10-15.json).
O build do job foi aprovado; este fechamento não exige rebuild ou novo restore.
Os tempos 0,9033 s/0,5403 s/0,4174 s são do worker, excluem startup e verificação
semântica no host; não constituem benchmark/SLA. Código e outputs históricos
permanecem preservados.

Gate de arquivamento/recuperação `gold_v1` fechado; leitores continuam locais.
Próximo incremento proposto: delimitar desenvolvimento e estudo Optuna, seus
vínculos e backup/restauração consistente do SQLite previsto no código. Modelos,
avaliações, MLflow e Helm/PVCs mantêm gates próprios. Nenhum dado/script removido.

[Runbook](../../docs/operations/STORAGE.md#gold-v1-completa-arquivo-com-contratos-e-protocolo) e
[evidência da preparação](../../references/evidence/gold_archive_preparation_2026-10-15.json).


## Desenvolvimento: dataset imutável e estudo transacional

O autor informou **64 arquivos/38.159.334 bytes de dataset** (manifesto e 63
Parquets) e **319 arquivos/232.943.522 bytes de estudo**. O SQLite tem 540.672
bytes; a presença de `.writer.lock` não comprova escritor ativo. A leitura foi
só de metadados. [Inventário](../../references/evidence/development_storage_inventory_author_2026-10-15.json).

A separação é por responsabilidade e recuperação: ambos cabem no teto atual.
Primeiro, o perfil `fraud_development_gold_bundle_v1` seleciona um run explícito,
manifesto/transactions e duas referências (inventário/protocolo HGB). Reutiliza
transporte e `verify_data` para identidade, política, datas, schema/features,
hashes, contagens e IDs únicos. Preserva bytes/run/proveniência; não recomputa
features ou consulta Silver, estudo, modelos ou MLflow. No restore, usa a cópia
e as referências baixadas. Código/outputs históricos permanecem preservados.

**Ciclo nativo aprovado pelo autor em 2026-10-15.** Preparo: 66 arquivos/
38.202.741 bytes com inventário e protocolo; publicação: 67 objetos enviados e
conferidos; reenvio: zero uploads/67 reutilizados; restore: `restored_verified`,
**63 partições, 19 features e 603.669 linhas** com política/inventário baixados.
Bundle `4e20595ffe482aacfb279d8dd8dea245e0ccc0f1d602c62f17d0cc52d12d6802`.
Manifesto de desenvolvimento preservado:
`b34988a71990356dd38ae62258d8f7ff232f9ecd8611fb9b9a31be80169d91e5`.
Sem ler originais, Bronze/Silver ou estudo, nem recomputar features; run e
`confirmation_evaluated=false` preservados. O autor reconstruiu o job e aprovou
seu hash; este fechamento não exige build ou novo restore. Os tempos do worker
são parciais e não medem o comando completo/verificação semântica no host.
[Evidência nativa](../../references/evidence/development_gold_archive_author_validation_2026-10-15.json).

Recibos em `data/storage/receipts/`: `development-gold-prepare-8o6sg3v7.json`,
`development-gold-publish-0w_sbr_y.json`, `development-gold-publish-qhkbfdl2.json`
e `development-gold-restore-762fn8gt.json`. O assistente recebeu as saídas
impressas, sem acessar dados/bucket/recibos nativos. Gate do dataset fechado.

O estudo continua no plano, com backup consistente do SQLite coordenado com
modelos/predições/tentativas/relatórios e vínculo por run ID, hash do manifesto e
bundle do dataset. O ciclo do estudo foi aprovado separadamente, conforme abaixo;
a cópia do dataset por si só não comprova recuperação do estudo ou de tracking. Leitores seguem locais.
[Runbook](../../docs/operations/STORAGE.md#gold-de-desenvolvimento-dataset-e-estudo) e
[preparação](../../references/evidence/development_gold_archive_preparation_2026-10-15.json).


## Estudo HGB: snapshot consistente com dependência explícita

O dataset recuperado é dependência explícita por bundle ID, run ID e hash do
manifesto. O snapshot coordenado com lock do escritor usa SQLite Online Backup
API e inclui modelos/predições/tentativas/recibos/relatórios; preserva inventário,
protocolo e manifesto do dataset, sem duplicar seus 63 Parquets. WAL/SHM/journal
e lock não integram o payload. SQLite tem equivalência lógica verificada.

O verificador existente reconcilia trials, métricas e artefatos em cópia
descartável com o dataset arquivado; adapta apenas `study_path` do relatório
dessa cópia. Relatório arquivado e referências de tracking ficam preservados.

**Ciclo S3 aprovado pelo autor em 2026-10-16:** 323 objetos publicados/verificados,
reenvio com zero uploads/323 reutilizados e restore de **322 arquivos/233.048.483
bytes**, preservando **20 trials concluídos, zero falhas, 63 tentativas**,
orçamento finalizado e decisão `retain_reference`. Integridade/foreign keys,
digest lógico SQLite e dependência foram conferidos. Fonte original não foi lida
no restore; modelos foram conferidos por hash, sem refit ou recarga/inferência.

| Identidade | Valor |
| --- | --- |
| Estudo | `e18ded1975bf541fe5993698f286c86c64e8abed514e085f64330905d7ec5198` |
| Dataset | `4e20595ffe482aacfb279d8dd8dea245e0ccc0f1d602c62f17d0cc52d12d6802` |
| Digest lógico SQLite | `c4e8397e1f1636800b45fe5cf842ce25a481df940ca5afbec62c4e691da5eee7` |

Bucket `fraud-artifacts`, prefixo `hgb-study/v1/<bundle_id>/`. Dependência local:
63 partições/603.669 linhas. Recibos em `data/storage/receipts/`:
`hgb-study-publish-ct9_zih6.json`, `hgb-study-publish-f90w4vod.json` e
`hgb-study-restore-m5t1puoi.json`. O assistente recebeu saídas impressas,
sem acessar banco/bucket/recibos nativos.
[Evidência S3](../../references/evidence/hgb_study_transfer_author_validation_2026-10-16.json).

O build foi aprovado. Este fechamento não exige reconstrução, novo snapshot ou
repetição do restore; o destino existente é protegido. O job usa `--no-deps`:
disponibilidade do RustFS deve ser conferida antes da transferência. A tentativa
inicial teve falha de conexão; após HTTP 200 o mesmo snapshot foi publicado.
Tempos do worker 4,9379 s/1,6495 s/1,6061 s excluem startup e verificação semântica
no host; não são tempo total/benchmark/SLA. Código e outputs históricos preservados.

Gate de recuperação do estudo concluído fechado; retomada de treinamento não foi
validada. Tracking não foi migrado e leitores continuam locais. PostgreSQL segue
escolhido para o servidor MLflow; SQLite mantém usos locais justificados.
Inventário MLflow/artefatos/URIs implementado, coletado e revisado pelo autor.
Preparo/verify de backup offline implementados/testados; primeira tentativa nativa
bloqueada pelo store de interface. Cobertura corrigida; retentativa de preparo e verificação nativos aprovadas. Recarga, transferência e migração permanecem como próximos gates. Helm/PVCs e cópia independente mantêm seus gates.
[Runbook](../../docs/operations/STORAGE.md#transferência-s3-do-estudo-hgb).


**Inventário do tracking MLflow — coleta corrigida e revisão do Default concluídas.**

Com UI MLflow e escritores de treino/avaliação parados, executar no terminal:
`poetry run python -m scripts.storage.mlflow_inventory`.
O comando lê arquivos do banco/WAL, consulta somente cópia temporária, levanta
schema/experimentos/runs/logged models, artefatos/checksums e referências JSON.
Grava relatório novo em data/storage/inventory e imprime resumo. Não inicializa
o banco original, carrega modelos, acessa rede ou migra serviços.

O código atual força SQLite e consumidores exigem mlflow.db. Antes da troca:
inventário nativo revisado, backup restaurável, transferência preservando IDs e
histórico, mapa de URIs S3, recarga/paridade e ensaio de rollback. PostgreSQL será
backend do servidor HTTP; esses gates ainda não foram implementados. Nenhum
novo fit ou uso de holdout é necessário para este inventário.
[Cobertura e procedimento](../../docs/operations/MLFLOW.md#inventário-antes-da-migração-postgresql--s3).


A coleta nativa inicial informou 5 experimentos, 70 runs FINISHED, 66 logged models,
627 arquivos/253.128.963 bytes e zero localizações gerenciadas ausentes. Dois
avisos de run_id eram UUIDs locais de diagnóstico, confundidos porque o relatório
também continha model_uri; essa heurística foi removida e testada. Nenhum recibo
científico foi alterado. O autor repetiu o inventário: zero referências não
resolvidas ou localizações gerenciadas ausentes. Default tem zero runs/modelos
e seu diretório não existe; aviso bruto review_required preservado, revisão
concluída. A migração continua pendente.


Coleta corrigida: `data/storage/inventory/mlflow-ljdjyua8.json`; 21 testes nativos,
um aviso, 7,31 s. A inspeção do Default usou associações do relatório inicial e
filesystem em separado, sem afirmar um snapshot atômico. Preservar o registro e
URI do Default, sem fabricar diretório ou ignorar futuras mudanças.
[Evidência nativa](../../references/evidence/mlflow_inventory_author_validation_2026-10-16.json).

A primeira tentativa nativa de backup parou no segundo store documentado,
`data/tracking/interface-smoke`: 1 run FINISHED, 1 logged model e 6 artefatos/32.088
bytes, sem localização gerenciada ausente. A correção inclui os dois bancos/WAL,
inventários e artefatos com verificação independente, referências conciliadas e
limites agregados de 1.024 arquivos/256 MiB. Não funde o histórico do exemplo com
o principal. Bancos desconhecidos continuam bloqueados. A retentativa de prepare
e verify nativos foi aprovada, assim como o ciclo S3. Recarga operacional,
PostgreSQL e inventário global continuam pendentes.

**Backup offline MLflow: preparo/verify implementados e testados; dois stores com preparo e verificação nativos aprovados.**
Com UI e escritores de ambos os stores parados, executar no terminal:
`poetry run python -m scripts.storage.mlflow_backup prepare --writers-stopped`.
O comando refaz inventário, captura SQLite/WAL em staging gerenciado, aplica
Online Backup API na cópia e inclui artefatos, auxiliares regulares e diretórios
com hashes e proveniência. Revalida o Default, orçamento e ausência de mutação
antes da publicação local. Não altera o banco original ou as URIs, carrega
modelos, acessa S3 ou provisiona PostgreSQL. Quiescência é declarada pelo operador,
sem alegação de lock global de escritores.

Perfil próprio: 1.024 arquivos/256 MiB, manifesto até 256 KiB. Arquivos/bytes e
IDs finais foram medidos no host, conforme a aprovação registrada abaixo. Saídas fixas em data/storage/bundles/<id> e
receipts; staging temporário também em data/storage, com remoção ao terminar.
`verify --bundle-id <id>` lê só a cópia, sem consultar paths históricos. A prova
com fontes removidas foi feita em fixtures, não no host do autor.
[Procedimento](../../docs/operations/MLFLOW.md#backup-offline-do-tracking-preparo-e-verificação).

O fechamento global exige inventariar todos os produtores, dados/cópias, volumes
e escritas de bibliotecas; corrigir destinos e retirar paths antigos após recuperar
e verificar a substituição. Isso permanece aberto; o backup não equivale à migração
primária nem à limpeza completa. [Contrato de persistência](../../docs/operations/STORAGE.md#critério-de-fechamento-da-persistência-global).

Regressão selecionada no tox py314: 123 passed, 1 skipped e 27 warnings; 12 casos novos para os dois stores.


**MLflow — preparo/verify nativos aprovados em 2026-10-16.**
Tox no host: 70 passed/1 warning, Ruff, formatação e lockfile aprovados.
O bundle `c9c4e6e0f684ba21484e7491c026a24a76002570c251c97c98b244f043a4d924` teve
mesmos hashes/contagens nas saídas de prepare e verify: 2 stores, 71 runs,
67 modelos, 633 artefatos; 640 arquivos/256.354.588 bytes no pacote. Integridade,
FKs, tabelas e URIs conferidas; verify informou original_source_read=false.
O assistente conciliou as saídas de terminal, sem acesso aos bancos/recibos completos.
O exemplo de interface permanece separado do histórico principal.

Transferência S3 implementada/testada usando storage-transfer, com manifesto
por último, readback integral e recuperação em data/storage/restored/<id> após
verificação dos dois stores e reconstrução dos diretórios vazios arquivados.
Bucket fraud-artifacts, prefixo mlflow-tracking/v1/<id>/; ciclo nativo aprovado
com 641 objetos verificados e 640 arquivos recuperados. Próximo passo: ensaio
operacional sobre cópia descartável do restore. Não refazer treino, remover originais ou reescrever URIs. Recarga
operacional MLflow, PostgreSQL, consumidores HTTP, rollback e inventário global
seguem abertos. Ver o runbook MLflow para os comandos.


**Ciclo S3 MLflow nativo aprovado em 2026-10-16.**
Build/readiness e tox (16 passed/1 warning) aprovados; publicação com 641 uploads,
reenvio com zero uploads/641 reutilizados e restore de 640 arquivos/256.354.588
bytes. Mesmos dois stores, 71 runs, 67 modelos, 633 artefatos e hashes físico/lógico
dos bancos, com integridade/FKs/tabelas/URIs aprovadas. O verificador não leu os
originais; os originais nativos não foram removidos. O assistente conciliou saídas
de terminal, sem acessar dados ou recibos completos. Gate de arquivamento S3 fechado.

Recarga MLflow implementada/testada com fixtures; execução nativa pendente. Antes de PostgreSQL, conferir histórico/modelo
em cópia descartável gerenciada do restore, identificando a referência e mapeando
paths apenas nessa cópia, sem ler artefatos originais. Bundle, restore validado,
fonte e evidências permanecem intactos. Não há nova avaliação, refit, promoção,
migração primária ou fechamento do inventário global neste passo.


## Recuperar histórico e modelo antes da migração

O ciclo S3 do snapshot está aprovado pelo autor. O novo ensaio usa seus dois
stores restaurados em cópia descartável, remapeia somente localizações de
artefatos e compara todas as células com essa transformação declarada. Pela API
MLflow, confere runs, parâmetros, tags, métricas e histórico; recarrega **um modelo**
fixado pela release de referência, após verificar seus bytes e revisar tipos skops.

A inferência de controle exige erro absoluto até `1e-12`. O worker tem telemetria
desativada, temporários/caches gerenciados e bloqueios Python para acesso a dados
originais, rede e processos adicionais. São controles verificáveis, sem promessa
de sandbox de sistema operacional. Restores são reconferidos e a cópia é removida;
o recibo local conserva o mapa. Nenhum novo serviço ou dependência foi adicionado.

**Implementado e testado com fixtures; validação nativa pendente.** O smoke prova
identidade/paridade, sem nova qualidade, servidor HTTP, PostgreSQL ou migração.
[Comando e limites](../../docs/operations/MLFLOW.md#ensaio-descartável-de-recuperação-mlflow).

O [contexto revisto](../../docs/project/PROBLEM_CONTEXT.md) separa ranking diário
retrospectivo da fila causal, explica denominadores e limites da simulação e
atualiza monitoramento, recuperação e prioridades. A infraestrutura atende a esses
controles; feature store é condicional. Inventário global, adoção pelos leitores,
rollback e limpeza continuam abertos.


### A execução nativa revelou uma diferença de tipos

Os 14 testes iniciais passaram no ambiente do autor. O ensaio nativo chegou a
`predict`, mas o MLflow recusou uma contagem `int64`: a assinatura arquivada exige
`double`. A fixture inferia tipos mistos do pedido HTTP e não reproduzia o caso.
O worker passa a converter inteiros apenas para destinos `double` declarados,
checando representação exata; valores que sofreriam arredondamento são recusados.
O modelo, assinatura arquivada, contratos, smoke e score de controle permanecem
preservados. O relatório registra as adaptações de tipo.

Regressão ampliada para assinaturas mistas/todas-double e para limites numéricos.
**Retentativa nativa pendente**, com o mesmo comando e os mesmos restores, sem
build/transferência/refit. [Detalhes](../../docs/operations/MLFLOW.md#compatibilidade-do-smoke-com-a-assinatura-histórica--correção-de-2026-10-16).


## Recuperação nativa MLflow concluída neste recorte

O autor informou 22 testes aprovados pelo tox e ensaio com `status=success`.
Pela API, os dois stores forneceram 71 runs, 67 logged models e 321 registros de
histórico de métricas. Uma referência foi recarregada; score esperado/observado
`0.4178630794049542`, erro zero. As 12 adaptações de inteiros para double seguiram
a assinatura sem perda de valor. Foram remapeadas 145 células de URI somente na
cópia descartável. Tabelas e restores preservados, cópia removida, fonte original
não lida. Os estados pendentes nas notas de preparação anteriores descrevem a
etapa antes dessa retentativa.

O resumo contou 1 operação de rede e 3 de processo bloqueadas, com zero operações
permitidas e zero violações de acesso aos dados. Os hooks são controles Python,
sem promessa de sandbox de sistema operacional. Recibo local:
`data/storage/receipts/mlflow-recovery-icz1xazs.json`. O assistente conciliou a saída,
sem receber os bancos, modelo ou recibo bruto.
[Resultado e limites](../../docs/operations/MLFLOW.md#recuperação-mlflow-validada-pelo-autor--2026-10-16).
[Evidência](../../references/evidence/mlflow_recovery_author_validation_2026-10-16.json).

Fechado o uso local pelo SDK em cópia SQLite; servidor HTTP/PostgreSQL, migração
primária, inventário global e limpeza permanecem abertos. Não há nova avaliação,
refit ou promoção. Este fechamento documental não exige repetir o ensaio.

A [introdução do contexto](../../docs/project/PROBLEM_CONTEXT.md) agora explica
o problema por um exemplo fictício e apresenta conceitos antes das métricas.
O exemplo de duas vagas distingue fila durante o dia e ranking retrospectivo.
Contratos, resultados e células executáveis não foram alterados.


## Inventário dos destinos antes da adoção pelos consumidores

A revisão encontrou um ponto cego: `archive inventory` cobre `data/`, excluindo
`data/storage/` e as saídas fora de `data/`. O novo coletor percorre o checkout
por metadados e associa arquivos a produtores/responsáveis do catálogo versionado.
Inclui bancos/sidecars como candidatos, sem abrir conteúdo ou executar produtores.
O relatório permanece em `data/storage/inventory/`, com permissão 0600.

É uma solução stdlib, sem serviço/dependência, que preserva os fingerprints dos
jobs de backup/transporte. 14 testes aprovados pelo tox; **coleta nativa pendente**.
Git/ambientes/caches de ferramentas são exclusões explícitas, com tamanho recursivo
desconhecido; caches em `data/` continuam incluídos. Links não são seguidos.

O catálogo é revisão manual, sem provar que todos os writers foram descobertos ou
confinados. Volumes Docker, bind `STATE_DIRECTORY`, argumentos de saída e caches
fora do checkout são fronteiras não inspecionadas. `project_wide_storage_inventory_complete`
continua falso. O scan não é snapshot consistente nem teste de recuperação.
Não houve migração ou remoção. O resultado nativo orientará os próximos destinos
PostgreSQL/S3 e a cobertura restante, antes de cutover/limpeza.
[Comando, catálogo e limites](../../docs/operations/STORAGE.md#inventário-do-checkout-e-catálogo-de-produtores).


### Inventário nativo e primeira contenção de destinos — 2026-10-16

O autor confirmou 4.990 arquivos/4.541.511.714 bytes lógicos e zero falhas de coleta.
O único item de revisão era o marcador vazio `data/external/.gitkeep`, preservado.
Os 20 candidatos a banco incluem originais, estados de replay e cópias arquivadas;
isso não autoriza deduplicar. Os mounts principais foram identificados: três volumes
nomeados e replay somente leitura no exporter; conteúdo/recuperação dos volumes
mantêm critérios próprios.

Quatro CLIs agora exigem destinos filhos de suas áreas em `data/`: contribuição
(examples e tracking), release sintética/recibo Docker (serving) e init da stack
(replay). A recusa ocorre antes de treino, Docker ou criação de estado. Relatórios
futuros usam `data/reports/`; nada foi movido. As funções internas e as demais CLIs
não estão contidas por este helper. Nenhum novo serviço, migração ou reavaliação.
Validação no assistente: 24 testes novos e seleção final de 82 testes aprovada no tox.
Validação nativa deste incremento pendente; global inventory permanece incompleto.

[Comandos e limites](../../docs/operations/STORAGE.md#destinos-de-escrita--primeira-cobertura-operacional).


### Tracking como serviço: PostgreSQL + RustFS — 2026-10-16

A contenção inicial teve 82 testes aprovados na máquina do autor. O próximo gate
é usar um serviço de tracking sem alterar os stores históricos: projeto Compose
`fraud-tracking-lab`, PostgreSQL com volume dedicado e bucket `fraud-tracking-lab/v1`
no RustFS existente. O MLflow faz proxy dos artefatos; o cliente não recebe suas
credenciais de armazenamento. A aplicação PostgreSQL usa papel sem superusuário.

O ensaio cria uma run operacional e um JSON pequeno; confere histórico/metadados
por HTTP e SQL, bytes por HTTP e S3; recria apenas os containers PostgreSQL/MLflow
e repete as leituras. A UI fica em `http://127.0.0.1:15001`. As 71 runs anteriores
não aparecem ali porque nenhuma migração foi feita. Isso não altera o modelo,
o dataset, a avaliação, os consumidores atuais ou a prontidão de produção.

Implementação e testes com fixtures preparados. O runtime/proxy HTTP real foi
conferido com SQLite/arquivos descartáveis; Docker/PostgreSQL/RustFS nativos ainda
aguardam execução pelo autor. Recibos ficam em `data/storage/tracking-lab/receipts/`.
O volume e o object store continuam no mesmo laptop; backup/restore PostgreSQL,
adoção de consumidores e Helm/PVCs exigem seus próprios critérios.

[Subir, validar, observar e parar](../../infra/tracking/README.md).


### Retentativa do tracking: contrato de saída do worker — 2026-10-16

PostgreSQL e MLflow subiram saudáveis na máquina do autor, com o bucket dedicado.
O ensaio parou depois da escrita/leitura HTTP do probe: o SDK imprimiu links antes
do JSON e o controlador recusou a saída. Ainda não foram aprovadas as conferências
independentes SQL/S3 nem a persistência após recriar os containers.

A correção envia mensagens do SDK para stderr e mantém um único JSON em stdout.
A regressão passa a executar o worker em processo separado contra HTTP real. Para
retomar, reconstruir/atualizar com `local_stack up` e repetir `tracking-lab`, sem
refazer `init`. A run anterior, o recibo da falha, credenciais, bucket e volume são
preservados; cada retentativa cria outra run operacional. Migração e adoção pelos
produtores continuam com seus próprios critérios.

[Procedimento e limites](../../infra/tracking/README.md).


### Tracking PostgreSQL + RustFS: persistência aprovada — 2026-10-17

O autor aprovou o ensaio nativo: a run
`87ac0b4a310f4ab8952a366932c0d12a` (experimento 2) foi conferida por SQL e HTTP;
o JSON foi lido pelo proxy e diretamente no RustFS. Após recriar os containers
PostgreSQL/MLflow, histórico de dois pontos e bytes permaneceram iguais. Os IDs
dos containers mudaram, com imagens e volume preservados; RustFS não reiniciou.
Pytest: **1 aprovado em 21,56 s**; tox `tracking-lab` OK. O recibo está em
`data/storage/tracking-lab/receipts/rehearsal-71acfcf3daca43348a02baa1d3cd27af.json`.

Isso conclui o gate que a primeira falha de JSON havia interrompido. A UI em
`http://127.0.0.1:15001` permite inspecionar a run, métrica e `probe.json`.
O assistente recebeu a saída impressa e conciliou os hashes do código/payload;
não executou os serviços nativos. O teste cobre uma run operacional e um JSON;
modelo pelo novo servidor, histórico, backup PostgreSQL, adoção pelos produtores
e Helm/PVCs mantêm critérios próprios.

Próximo recorte: ligar um produtor de novas runs via URI HTTP explícita, verificar
recarga de seu modelo pelo proxy e o caminho de rollback. Este fechamento
preserva todos os scripts, dados e registros anteriores.

[Operação](../../infra/tracking/README.md).
[Evidência](../../references/evidence/tracking_lab_author_validation_2026-10-17.json).


### Primeiro produtor HTTP: exemplo de contribuição — 2026-10-17

O laboratório preservou metadados/JSON após recriação dos containers. O novo
recorte usa o exemplo existente para publicar e recarregar um modelo skops pelo
proxy, com `--tracking-uri` explícita e experimento
`fraud-model-interface-smoke-http-v1`. São 80/20 linhas geradas em memória;
não é avaliação antifraude, consumo de holdout ou promoção de candidato.

A opção local continua disponível por `--tracking-root`, mutuamente exclusiva.
Falha HTTP encerra com erro e não redireciona para SQLite. URI e limites são
restaurados ao sair; rollback significa escolher o destino local numa nova
execução, preservando os registros remotos. As três saídas de inspeção ficam em
`data/examples/`; o recibo da prova nativa, em `data/storage/tracking-lab/receipts/`.

O teste HTTP descartável cobre a CLI, recarga sem a saída local, paridade,
indisponibilidade e retorno local. A prova desse produtor no PostgreSQL/RustFS
nativo está pendente e será executada por `tox -e tracking-contribution`, sem
reiniciar serviços. O teste preserva o exemplo HTTP e usa somente um store novo
local descartável para verificar rollback. Baseline, HGB e histórico mantêm seus
fluxos atuais.

[Contrato, comandos e aceite](../../docs/modeling/CONTRIBUTING_MODELS.md#primeiro-produtor-http-no-laboratório).


### Correção de transporte HTTP após a primeira tentativa nativa — 2026-10-17

Os 79 testes de regressão passaram. O produtor nativo chegou à run e à listagem
dos cinco arquivos do modelo, mas falhou no download por chunks. Recarga e
rollback local continuam pendentes. O MLflow 3.17 pode selecionar URLs assinadas
automaticamente quando o S3 anuncia essa capacidade; o fixture com backend de
arquivos não cobria esse comportamento.

O contexto HTTP agora fixa as opções de multipart por proxy em `false`, para que
os bytes atravessem o servidor MLflow. URI, limites e flags anteriores são
restaurados ao sair. O novo teste usa SDK/servidor reais com capacidades multipart
anunciadas e destino direto inacessível: reproduziu a falha antes da correção e
exige sucesso, paridade e GET pelo proxy, sem pedidos de URLs assinadas.

A relação com `rustfs:9000` é inferida da configuração; as URLs assinadas nativas
não foram coletadas. Não houve ajuste de timeout, troca de endpoint, rebuild ou
alteração do treino. A run/recibo da falha são preservados. Repetir
`tox -e tracking-contribution` após aplicar a correção gera uma nova contribuição;
o aceite PostgreSQL/RustFS ainda depende dessa retentativa.

[Diagnóstico e retentativa](../../infra/tracking/README.md#retentativa-do-produtor-após-falha-de-download-por-chunks).


### Contribuição HTTP aprovada e próximo produtor: HGB

Em 2026-10-17, o autor aprovou 34 testes de regressão e 1 teste nativo da
contribuição. O modelo foi recarregado pelo HTTP sem a saída local disponível,
com erro zero; o retorno local foi explícito e os containers permaneceram iguais.

O próximo gate é um **fold HGB sintético**, pelo mesmo serviço: endpoint vinculado
à identidade, recarga independente e reutilização sem treino/publicação repetidos.
O teste nativo ainda está pendente. Não repetir o estudo de 20 trials/63 tentativas
ou as janelas já consumidas. O estado Optuna permanece SQLite; PostgreSQL é o
backend do MLflow compartilhado. A otimização completa preserva o gate de código
e política commitados; o ensaio sintético não o altera.

[Comandos, persistência e limites](../../docs/operations/HGB_TRACKING.md).


### Fold HGB aprovado; adoção HTTP do trio inicial

Em 2026-10-17, o autor aprovou o fold sintético do runner Optuna HGB: 84 linhas
de treino, 21 de validação, uma tentativa, recarga independente com erro zero e
reutilização sem novo fit/publicação. Os containers foram preservados.

Isso não incluía o trio inicial de baselines, que é outro runner. A nova opção
HTTP abrange **dummy, regressão logística e HGB**, com uma run por modelo e
manifest/audit em área inventariada. O aceite nativo do trio está pendente e usa
160/160 linhas sintéticas. Não repetir Handbook, otimização ou janelas consumidas.
O local continua disponível, e a migração histórica não foi implementada.

[Procedimento, falhas parciais e limites](../../docs/modeling/BASELINE.md#tracking-http-do-trio-de-baselines).


### Trio de baselines aprovado pelo autor — 2026-10-17

O ensaio nativo terminou com sucesso: três runs independentes para dummy,
regressão logística e HGB; 160 linhas de treino e 160 de validação sintéticas.
Os modelos foram recarregados pelo HTTP com os arquivos locais indisponíveis,
e o erro máximo foi zero em todos. Não houve tracking SQLite novo ou alteração
dos containers. Pytest: 1 aprovado em 6,55 s; tox OK em 19,96 s.

O aceite desta integração está concluído. Contribuição e fold Optuna HGB têm
suas próprias evidências. O histórico e o padrão local permanecem; consumidores
e recuperação PostgreSQL/S3 ainda precisam de planejamento antes de cutover.
Não é preciso repetir treino, ampliar serviços ou limpar dados para registrar
o resultado. [Evidência do autor e limites](../../references/evidence/baseline_tracking_author_validation_2026-10-17.json).


### Revisão dos consumidores e protocolo de recuperação

Depois dos aceites HTTP dos produtores, foram revisados 17 registros de fontes.
Sete fronteiras continuam locais: ferramentas de histórico, diagnósticos,
ablações, freeze, avaliação final, avaliação estatística e exportação. IDs,
hashes, ownership e ledger não podem ser substituídos por uma troca global de URI.
A inferência da release já é independente do tracking.

O coletor existente recebe `--tracking-consumers`: confere fontes/funções do
catálogo sem importar módulos, ler dados/bancos/modelos ou consultar serviços.
No checkout preparado, 124 fontes e 55 chamadas sintáticas, revisão válida.
O relatório nativo ainda será observado; não é inventário global.

O próximo ensaio foi delimitado: captura coordenada de PostgreSQL/S3 e restore
num alvo próprio, com histórico e modelo/score fixados. Seu job ainda não foi
implementado. Consumidores científicos e infraestrutura atual foram preservados.
[Comandos e protocolo](../../docs/operations/TRACKING_RECOVERY.md).


### Captura coordenada do tracking — 2026-10-19

O autor confirmou o inventário: 48 testes, 17 registros, sete fronteiras locais,
zero divergências. A captura PostgreSQL/S3 da bancada foi implementada e testada
sem serviços; seu gate nativo está pendente. O job adquire lock, pausa apenas o
MLflow, faz dump custom e copia a namespace S3, confere hashes e tenta devolver
o container original à saúde antes de declarar sucesso. Tudo permanece em
`data/storage/tracking-lab/captures` e `receipts`, com identidade explícita.

Não repetir treinos ou avaliações para testar infraestrutura. O dump e a cópia
são pré-requisitos da próxima prova: restaurar PostgreSQL e RustFS em volumes
separados e recarregar o modelo fixado sem fallback. Esse restore ainda não foi
implementado. `postgres_backup_restore_validated=false` continua correto.

Os comandos explícitos e os limites estão no runbook `TRACKING_RECOVERY.md`.
Nenhuma célula anterior foi reexecutada; evidências e saídas históricas preservadas.


### Fechamento do recorte de inventário/recuperação — gate final

Captura aprovada pelo autor: ID `5a2feb3556834dcfa73d14072bdc5182`, 38 objetos,
8 runs, 6 logged models, 44 pontos históricos e dump de 149.587 bytes. Origem
saudável e identidades preservadas; não repetir captura ou treinamento.

O restore isolado foi implementado: rede e volumes PostgreSQL/RustFS novos,
comparação lógica/estrutural dos dados, API com histórico completo e recarga do
modelo da contribuição sintética contra scores salvos antes do alvo existir.
O ensaio remove seus recursos temporários; conserva controles e recibo. Seus
contratos offline e recarga HTTP real foram testados; o gate Docker nativo ainda
está pendente. Ver `TRACKING_RECOVERY.md` para build e `tox -e tracking-restore`.

Esse aceite encerra este recorte. Migração dos sete consumidores históricos,
cutover e retenção/limpeza são backlog separado, não motivo para prolongar
inventário antes de CD/rollback e monitoramento de features. Inventário global,
adoção primária e produção continuam sem validação. Células históricas preservadas.


### Diagnóstico de falha do restore — 2026-10-19

O ensaio nativo chegou ao worker de reposição/conferência após pg_restore, mas
sua mensagem foi ocultada pelo wrapper. O gate continua pendente; a causa ainda
não foi determinada. O worker agora retorna falha estruturada, preservada no
recibo e mostrada pelo controller, com fase, código seguro e diferenças de
metadados por nomes/hashes. Não inclui valores de tabelas, URLs ou senhas.
A captura aprovada e os protocolos científicos permanecem intactos.


### Restore: diferença de representação do esquema (2026-10-19)

A reprodução PostgreSQL 17.11 / MLflow 3.17.0 coincidiu com os quatro hashes nativos. O dump compacta uma lacuna de coluna removida em `spans` e reescreve cinco checks de enumeração com casts equivalentes. O worker compara a ordem relativa das colunas existentes e somente esses cinco pares exatos; tipos, defaults, nulabilidade, demais regras e dados continuam comparados. Testes com alterações SQL reais reprovaram. Captura original preservada. O gate nativo completo ainda aguarda nova execução; não há migração, retreino ou cutover.


### Recorte de inventário e recuperação encerrado — 2026-10-19

O autor aprovou o restore PostgreSQL/S3 isolado: **38 objetos, 8 runs, 6 modelos e 44 pontos históricos**. A API conciliou o histórico, o modelo fixado recarregou pelo proxy e reproduziu **20 scores com erro máximo zero**. Fonte e captura preservadas; alvo e credenciais temporários removidos. Pytest live: 1 aprovado em 20,89 s; SDK: 56 aprovados e 1 caso opcional pulado.

A evidência é uma saída nativa fornecida pelo autor; não houve reexecução de Docker pelo assistente. O marco fecha este recorte laboratorial, sem migração do histórico SQLite, cutover ou inventário global. Não repetir captura/treino/restore para este fechamento. **Próxima prioridade: CD e rollback do serving; depois, monitoramento de dados/features.** Consumidores históricos e limpeza ficam em backlog próprio.

[Evidência do autor](../../references/evidence/tracking_restore_author_validation_2026-10-19.json).
